In [57]:
import os
import sys
import pandas as pd
import osmnx as ox

# ============
# CONFIGURACIÓN
# ============

# Ruta base si usas config externa
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from config import DATA_EXTERNAL_OSM_DIR

# Tipos de lugares que queremos extraer de OSM
tipos_osm = {
    'amenity': [
        'school', 'kindergarten', 'university', 'college', 'hospital', 'clinic', 'pharmacy',
        'library', 'restaurant', 'cafe', 'bar', 'bank', 'atm', 'supermarket', 'convenience',
        'place_of_worship', 'theatre', 'cinema', 'museum', 'community_centre', 'childcare'
    ],
    'shop': True,
    'leisure': ['park', 'playground', 'sports_centre', 'pitch'],
    'public_transport': True,
    'railway': ['station', 'subway_entrance'],
    'highway': ['bus_stop'],
    'natural': ['wood', 'water', 'beach'],
    'landuse': ['industrial', 'commercial']
}

# Traducción de tipos al español
traduccion_tipo = {
    'station': 'Estación',
    'stop_position': 'Paradero',
    'bus_stop': 'Paradero de bus',
    'subway_entrance': 'Entrada de metro',
    'school': 'Escuela',
    'kindergarten': 'Jardín',
    'university': 'Universidad',
    'college': 'Instituto',
    'childcare': 'Guardería',
    'hospital': 'Hospital',
    'clinic': 'Clínica',
    'pharmacy': 'Farmacia',
    'doctors': 'Consultorio',
    'supermarket': 'Supermercado',
    'convenience': 'Tienda',
    'restaurant': 'Restaurante',
    'cafe': 'Café',
    'bar': 'Bar',
    'place_of_worship': 'Templo',
    'library': 'Biblioteca',
    'cinema': 'Cine',
    'theatre': 'Teatro',
    'museum': 'Museo',
    'park': 'Parque',
    'playground': 'Juegos',
    'sports_centre': 'Deporte',
    'pitch': 'Cancha',
    'bank': 'Banco',
    'atm': 'Cajero',
    'community_centre': 'Centro comunitario',
    'wood': 'Bosque',
    'water': 'Agua',
    'beach': 'Playa',
    'industrial': 'Zona industrial',
    'commercial': 'Zona comercial'
}

# Grupos generales
# grupo_funcional = {
#     'Educación': ['school', 'kindergarten', 'university', 'college', 'childcare'],
#     'Salud': ['hospital', 'clinic', 'pharmacy', 'doctors'],
#     'Comercio': ['shop', 'supermarket', 'convenience'],
#     'Restauración': ['restaurant', 'cafe', 'bar'],
#     'Religioso': ['place_of_worship', 'community_centre'],
#     'Cultura y recreación': ['library', 'cinema', 'theatre', 'museum', 'park', 'playground', 'sports_centre', 'pitch'],
#     'Transporte': ['station', 'bus_stop', 'subway_entrance', 'stop_position'],
#     'Financiero': ['bank', 'atm'],
#     'Zona urbana': ['industrial', 'commercial'],
#     'Naturaleza': ['wood', 'water', 'beach']
# }

grupo_funcional = {
    'educ': [  # Educación
        'school', 'kindergarten', 'university', 'college', 'childcare'
    ],
    'salud': [  # Servicios de salud
        'hospital', 'clinic', 'pharmacy', 'doctors'
    ],
    'com_serv': [  # Comercio y servicios básicos
        'shop', 'supermarket', 'convenience'
    ],
    'finenciero': [  # financiero
        'bank', 'atm'
    ],
    'aliment_ocio': [  # Alimentación y ocio
        'restaurant', 'cafe', 'bar'
    ],
    'cult_recre': [  # Cultura y recreación
        'library', 'cinema', 'theatre', 'museum', 'park', 'playground', 'sports_centre', 'pitch'
    ],
    'transp_pub': [  # Transporte público
        'station', 'bus_stop', 'subway_entrance', 'stop_position'
    ],
    'social_com': [  # Centros sociales o comunitarios
        'place_of_worship', 'community_centre'
    ],
    'zona_urb': [  # Zonas urbanas (comercio/industria)
        'industrial', 'commercial'
    ],
    'entorno_nat': [  # Naturaleza o entorno ambiental
        'wood', 'water', 'beach'
    ]
}



# ============
# EXTRACTOR
# ============

class ExtraccionOSM:
    def __init__(self, ruta_excel):
        self.ruta_excel = ruta_excel
        os.makedirs(os.path.dirname(self.ruta_excel), exist_ok=True)

    def obtener_lugares(self, ciudad, filtros_osm):
        zona = ox.geocode_to_gdf(ciudad)
        poligono = zona.geometry.iloc[0]
        lugares = ox.features_from_polygon(poligono, tags=filtros_osm)
        return lugares

    def procesar_y_guardar(self, lugares):
        registros = []

        columnas_utiles = [col for col in lugares.columns if col in [
            'amenity', 'shop', 'leisure', 'public_transport',
            'railway', 'highway', 'natural', 'landuse'
        ]]

        for col in columnas_utiles:
            sub = lugares[[col, 'geometry']].dropna()
            sub = sub[sub.geometry.type == 'Point']
            for _, row in sub.iterrows():
                tipo_en = row[col]
                tipo_es = traduccion_tipo.get(tipo_en, tipo_en)
                grupo = self.obtener_grupo(tipo_en)

                registros.append({
                    'lat': row.geometry.y,
                    'lon': row.geometry.x,
                    'tipo_osm': tipo_en,
                    'etiqueta': col,
                    'tipo': tipo_es,
                    'grupo': grupo
                })

        df = pd.DataFrame(registros).drop_duplicates()
        df.to_excel(self.ruta_excel, index=False)
        print(f"✅ Archivo guardado en {self.ruta_excel}")

    def obtener_grupo(self, tipo):
        for grupo, lista in grupo_funcional.items():
            if tipo in lista:
                return grupo
        return 'Otro'


# ============
# USO
# ============

ruta_salida = os.path.join(DATA_EXTERNAL_OSM_DIR, 'OpenStreetMap.xlsx')
osm = ExtraccionOSM(ruta_excel=ruta_salida)

data_osm = osm.obtener_lugares("Melbourne, Australia", filtros_osm=tipos_osm)
osm.procesar_y_guardar(data_osm)

✅ Archivo guardado en c:\Users\carlo\OneDrive\U\1er\introds\proyectofinal\data\external\osm_places\OpenStreetMap.xlsx


In [58]:
# import os
# import osmnx as ox
# import pandas as pd
# from shapely.geometry import Point

# class OpenStreetMapExtractor:
#     def __init__(self, excel_path):
#         self.excel_path = excel_path
#         os.makedirs(os.path.dirname(self.excel_path), exist_ok=True)

#     def extract_from_polygon(self, place_name, tags):
#         # Obtener polígono del lugar (ej. "Melbourne, Australia")
#         area = ox.geocode_to_gdf(place_name)
#         polygon = area.geometry.iloc[0]

#         # Extraer features dentro del polígono
#         gdf = ox.features_from_polygon(polygon, tags=tags)
#         return gdf

#     def procesar_y_guardar(self, gdf):
#         resultados = []
#         for col in ['amenity', 'shop', 'leisure', 'public_transport']:
#             if col in gdf.columns:
#                 sub = gdf[[col, 'geometry']].dropna()
#                 sub = sub[sub.geometry.type == 'Point']  # Filtrar solo puntos válidos
#                 for _, row in sub.iterrows():
#                     value = row.get(col)
#                     resultados.append({
#                         'Lattitude': row.geometry.y,
#                         'Longtitude': row.geometry.x,
#                         'values': value if value else col
#                     })
                    
#         df_resultado = pd.DataFrame(resultados)
#         df_resultado.drop_duplicates(inplace=True)
#         df_resultado.to_excel(self.excel_path, index=False)
#         print(f"✅ Resultados guardados en {self.excel_path}")

In [59]:
# import sys
# import os
# project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
# if project_root not in sys.path:
#     sys.path.append(project_root)
# from config import *

# # Este diccionario 'tags' define qué tipos de establecimientos o elementos queremos extraer desde OpenStreetMap.
# # Se utiliza para enriquecer el análisis geográfico de precios de propiedades, extrayendo puntos de interés
# # cercanos como servicios, transporte, comercio y espacios verdes.

# tags = {
#     # Servicios y equipamientos públicos o privados
#     'amenity': [
#         'school',            # Escuela
#         'kindergarten',      # Jardín de infancia
#         'university',        # Universidad
#         'college',           # Instituto
#         'hospital',          # Hospital
#         'clinic',            # Clínica
#         'pharmacy',          # Farmacia
#         'library',           # Biblioteca
#         'restaurant',        # Restaurante
#         'cafe',              # Café
#         'bar',               # Bar
#         'bank',              # Banco
#         'atm',               # Cajero automático
#         'supermarket',       # Supermercado
#         'convenience',       # Tienda de conveniencia
#         'place_of_worship',  # Lugar de culto (iglesia, templo, etc.)
#         'theatre',           # Teatro
#         'cinema',            # Cine
#         'museum',            # Museo
#         'community_centre',  # Centro comunitario
#         'childcare'          # Guardería
#     ],

#     # Todas las tiendas (ropa, panaderías, tecnología, etc.)
#     'shop': True,

#     # Espacios recreativos
#     'leisure': [
#         'park',             # Parque
#         'playground',       # Juegos infantiles
#         'sports_centre',    # Centro deportivo
#         'pitch'             # Cancha (de fútbol, tenis, etc.)
#     ],

#     # Infraestructura de transporte público
#     'public_transport': True,

#     # Estaciones de tren y entradas de metro
#     'railway': [
#         'station',          # Estación de tren
#         'subway_entrance'   # Entrada de metro
#     ],

#     # Paraderos de buses
#     'highway': [
#         'bus_stop'          # Paradero de bus
#     ],

#     # Elementos naturales
#     'natural': [
#         'wood',             # Bosques
#         'water',            # Cuerpos de agua (ríos, lagos)
#         'beach'             # Playa
#     ],

#     # Zonas de uso del suelo
#     'landuse': [
#         'industrial',       # Zonas industriales
#         'commercial'        # Zonas comerciales
#     ]
# }

# excel_path = os.path.join(DATA_EXTERNAL_OSM_DIR, 'OpenStreetMap.xlsx')
# extractor = OpenStreetMapExtractor(excel_path=excel_path)

# gdf = extractor.extract_from_polygon("Melbourne, Australia", tags=tags)
# extractor.procesar_y_guardar(gdf)

| Tag / Grupo         | ¿Qué representa?                                                             | ¿Para qué sirve?                                                                                                       |
| ------------------- | ---------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------- |
| `amenity`           | Servicios como escuelas, hospitales, bancos, restaurantes, bibliotecas, etc. | Permiten identificar qué tan equipada está una zona en términos de educación, salud, alimentación y servicios básicos. |
| `shop: True`        | Cualquier tipo de tienda (supermercados, panaderías, ropa, etc.).            | Ayuda a evaluar la conveniencia de la zona para hacer compras.                                                         |
| `leisure`           | Lugares recreativos como parques, centros deportivos, áreas de juego.        | La cercanía a espacios de recreación incrementa el atractivo de una propiedad.                                         |
| `public_transport`  | Infraestructura de transporte público general.                               | Mejora la conectividad y acceso, impactando positivamente el valor de la vivienda.                                     |
| `railway`           | Estaciones de tren y metro.                                                  | Específicamente útil para modelar acceso al transporte masivo.                                                         |
| `highway: bus_stop` | Paraderos de buses.                                                          | Complementa el análisis de conectividad local.                                                                         |
| `natural`           | Elementos naturales como bosques, agua y playas.                             | Zonas cercanas a naturaleza suelen ser más valorizadas.                                                                |
| `landuse`           | Uso del suelo como industrial o comercial.                                   | Puede ser positivo (zona comercial) o negati                                                                           |


In [60]:
import pandas as pd
osm_places = pd.read_excel(os.path.join(DATA_EXTERNAL_OSM_DIR, 'OpenStreetMap.xlsx'))

In [66]:
osm_places.grupo.value_counts()

grupo
Otro            25193
transp_pub      15835
aliment_ocio     6031
cult_recre       1394
com_serv         1370
salud             902
finenciero        754
social_com        544
educ              433
entorno_nat        16
zona_urb            9
Name: count, dtype: int64

In [ ]:
# import pandas as pd

# mapa_grupos = {
#     # Educación
#     'school': 'Educación',
#     'kindergarten': 'Educación',
#     'university': 'Educación',
#     'college': 'Educación',
#     'childcare': 'Educación',

#     # Salud
#     'hospital': 'Salud',
#     'clinic': 'Salud',
#     'pharmacy': 'Salud',
#     'doctors': 'Salud',

#     # Comercio y servicios
#     'shop': 'Comercio y servicios',
#     'fuel': 'Comercio y servicios',
#     'fast_food': 'Comercio y servicios',
#     'winery': 'Comercio y servicios',
#     'ice_cream': 'Comercio y servicios',
#     'internet_cafe': 'Comercio y servicios',
#     'bureau_de_change': 'Comercio y servicios',
#     'post_office': 'Comercio y servicios',
#     'post_box': 'Comercio y servicios',

#     # Restauración y vida social
#     'restaurant': 'Restauración y vida social',
#     'cafe': 'Restauración y vida social',
#     'bar': 'Restauración y vida social',

#     # Espiritual y comunitario
#     'place_of_worship': 'Espiritual y comunitario',
#     'community_centre': 'Espiritual y comunitario',

#     # Cultura y recreación
#     'library': 'Cultura y recreación',
#     'theatre': 'Cultura y recreación',
#     'cinema': 'Cultura y recreación',
#     'leisure': 'Cultura y recreación',
#     'skatepark': 'Cultura y recreación',

#     # Transporte y conectividad
#     'public_transport': 'Transporte y conectividad',
#     'bus_station': 'Transporte y conectividad',
#     'ferry_terminal': 'Transporte y conectividad',
#     'bicycle_parking': 'Transporte y conectividad',
#     'shelter': 'Transporte y conectividad',

#     # Financiero
#     'bank': 'Financiero',
#     'atm': 'Financiero'
# }

# # Asignar grupo funcional
# osm_places['grupo_funcional'] = osm_places['values'].map(mapa_grupos).fillna('Otro')

In [67]:
osm_places.to_excel(os.path.join(DATA_EXTERNAL_OSM_DIR, 'OpenStreetMap.xlsx'), index=False)